In [1]:
import pandas as pd
import numpy as np
import math, time
import matplotlib.pyplot as plt
from utils.preprocess import train_test_split
from utils.ticker_data import windowed_dfs
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import mean_absolute_percentage_error

In [2]:
def prepare_split(ticker, timeframe, train_window=30):
    df = pd.read_csv(f'results/train/data/{timeframe}/{ticker}.csv')
    if 'Date' in df:
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.set_index('Date')
    else:
        df['Datetime'] = pd.to_datetime(df['Datetime'])
        df = df.set_index('Datetime')
    scaler = StandardScaler()
    df['Close'] = scaler.fit_transform(df[['Close']])
    X, y = windowed_dfs(df, columns=['Close'], target='Close', train_window=train_window)
    X_train, y_train, X_test, y_test, dates_test = train_test_split(X, y, df.index)
    return X_train, y_train, X_test, y_test

In [3]:
def grid_search_weighted_mape(estimator, param_grid, tickers):
    tot, curr = math.prod([len(v) for v in param_grid.values()]), 1
    best_params, best_score = None, np.inf

    start_all = time.time()
    for params in ParameterGrid(param_grid):
        start = time.time()
        total_error, total_n = 0.0, 0

        for ticker in tickers:
            for timeframe in ['daily','hourly']:
                X_tr, y_tr, X_te, y_te = prepare_split(ticker, timeframe)

                model = estimator(**params)
                model.fit(X_tr, y_tr)
                y_pred = model.predict(X_te)

                mape_i = mean_absolute_percentage_error(y_te, y_pred)
                n_i = len(y_te)

                total_error += mape_i * n_i
                total_n += n_i

        if total_n > 0:
            weighted_mape = total_error / total_n
            elapsed = time.time() - start
            print(f"[{curr}/{tot}] {params} - MAPE: {weighted_mape:.4f} (time: {elapsed:.1f}s)")

            if weighted_mape < best_score:
                best_score  = weighted_mape
                best_params = params

        curr += 1

    total_elapsed = time.time() - start_all
    print(f"\nGrid search completed in {total_elapsed/60:.2f} minutes")
    print("Best params:", best_params)
    print("Best weighted MAPE:", best_score)

    return best_params, best_score

tickers = ['EMBASSY.BO', 'GODREJPROP.NS', '^GSPC', 'GOOGL']

In [ ]:
param_grid = {'n_neighbors': [3, 5, 7, 10, 15], 'weights': ['distance', 'uniform'], 'p': [1, 2], 'metric': ['minkowski', 'chebyshev']}
grid_search_weighted_mape(
    estimator=KNeighborsRegressor,
    param_grid=param_grid,
    tickers=tickers
) 
# Grid search completed in 2.00 minutes
# Best params: {'metric': 'minkowski', 'n_neighbors': 15, 'p': 1, 'weights': 'uniform'}
# Best weighted MAPE: 1.2486844365651992

[1/40] {'metric': 'minkowski', 'n_neighbors': 3, 'p': 1, 'weights': 'distance'} - MAPE: 1.6387 (time: 2.6s)
[2/40] {'metric': 'minkowski', 'n_neighbors': 3, 'p': 1, 'weights': 'uniform'} - MAPE: 1.6380 (time: 2.7s)
[3/40] {'metric': 'minkowski', 'n_neighbors': 3, 'p': 2, 'weights': 'distance'} - MAPE: 1.7291 (time: 2.1s)
[4/40] {'metric': 'minkowski', 'n_neighbors': 3, 'p': 2, 'weights': 'uniform'} - MAPE: 1.7286 (time: 2.1s)
[5/40] {'metric': 'minkowski', 'n_neighbors': 5, 'p': 1, 'weights': 'distance'} - MAPE: 1.5800 (time: 2.4s)
[6/40] {'metric': 'minkowski', 'n_neighbors': 5, 'p': 1, 'weights': 'uniform'} - MAPE: 1.5780 (time: 2.7s)
[7/40] {'metric': 'minkowski', 'n_neighbors': 5, 'p': 2, 'weights': 'distance'} - MAPE: 1.5293 (time: 2.5s)
[8/40] {'metric': 'minkowski', 'n_neighbors': 5, 'p': 2, 'weights': 'uniform'} - MAPE: 1.5265 (time: 2.7s)
[9/40] {'metric': 'minkowski', 'n_neighbors': 7, 'p': 1, 'weights': 'distance'} - MAPE: 1.7026 (time: 2.8s)
[10/40] {'metric': 'minkowski', 

({'metric': 'minkowski', 'n_neighbors': 15, 'p': 1, 'weights': 'uniform'},
 1.2486844365651992)

In [ ]:
param_grid_xgb = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'gamma': [0, 1, 5]
}
grid_search_weighted_mape(XGBRegressor, param_grid_xgb, tickers)

# Grid search completed in 65.55 minutes
# Best params: {'colsample_bytree': 1.0, 'gamma': 1, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 300, 'subsample': 0.6}
# Best weighted MAPE: 0.24698669218287822

[1/729] {'colsample_bytree': 0.6, 'gamma': 0, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.6} - MAPE: 0.7039 (time: 5.2s)
[2/729] {'colsample_bytree': 0.6, 'gamma': 0, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8} - MAPE: 0.6666 (time: 4.7s)
[3/729] {'colsample_bytree': 0.6, 'gamma': 0, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100, 'subsample': 1.0} - MAPE: 0.6251 (time: 4.7s)
[4/729] {'colsample_bytree': 0.6, 'gamma': 0, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.6} - MAPE: 0.6467 (time: 7.0s)
[5/729] {'colsample_bytree': 0.6, 'gamma': 0, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.8} - MAPE: 0.6010 (time: 6.9s)
[6/729] {'colsample_bytree': 0.6, 'gamma': 0, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200, 'subsample': 1.0} - MAPE: 0.5813 (time: 7.0s)
[7/729] {'colsample_bytree': 0.6, 'gamma': 0, 'learning_rate': 0.01, 'max_depth': 3, 'n_

({'colsample_bytree': 1.0,
  'gamma': 1,
  'learning_rate': 0.1,
  'max_depth': 5,
  'n_estimators': 300,
  'subsample': 0.6},
 0.24698669218287822)

In [ ]:
param_grid_lgbm = {
    'n_estimators': [100, 200, 300],
    'num_leaves': [31, 63, 127],
    'learning_rate': [0.01, 0.05, 0.1],
    'feature_fraction': [0.6, 0.8, 1.0],
    'bagging_fraction': [0.6, 0.8, 1.0],
    'min_child_samples': [5, 10, 20],
    'verbose':[-1]
}
grid_search_weighted_mape(LGBMRegressor, param_grid_lgbm, tickers)

# Best params:{'bagging_fraction': 0.6, 'feature_fraction': 1.0, 'learning_rate': 0.1, 'min_child_samples': 5, 'n_estimators': 100, 'num_leaves': 63, 'verbose': -1}
# Best weighted MAPE: 0.34384053678953896

[1/729] {'bagging_fraction': 0.6, 'feature_fraction': 0.6, 'learning_rate': 0.01, 'min_child_samples': 5, 'n_estimators': 100, 'num_leaves': 31, 'verbose': -1} - MAPE: 0.8777 (time: 3.2s)
[2/729] {'bagging_fraction': 0.6, 'feature_fraction': 0.6, 'learning_rate': 0.01, 'min_child_samples': 5, 'n_estimators': 100, 'num_leaves': 63, 'verbose': -1} - MAPE: 0.7675 (time: 3.9s)


KeyboardInterrupt: 

In [ ]:
param_grid_rf = {
    'n_estimators': [100, 200, 500],
    'max_depth': [None, 10, 20, 30],
    'max_features': ['sqrt', 'log2', 0.6, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
grid_search_weighted_mape(RandomForestRegressor, param_grid_rf, tickers)

# Grid search completed in 1003.62 minutes
# Best params: {'max_depth': None, 'max_features': None, 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 100}
# Best weighted MAPE: 0.2619929509980757

[1/432] {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100} - MAPE: 0.6913 (time: 22.7s)
[2/432] {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200} - MAPE: 0.6337 (time: 43.0s)
[3/432] {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 500} - MAPE: 0.6673 (time: 102.2s)
[4/432] {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100} - MAPE: 0.6685 (time: 18.9s)
[5/432] {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200} - MAPE: 0.6911 (time: 36.6s)
[6/432] {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 500} - MAPE: 0.6549 (time: 93.3s)
[7/432] {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_es

({'max_depth': None,
  'max_features': None,
  'min_samples_leaf': 2,
  'min_samples_split': 10,
  'n_estimators': 100},
 0.2619929509980757)

polynomial kernels are ridiculously bad for this task, and only works for vey high degrees, might as well not train them at all.

In [ ]:
# param_grid_svr = {
#     'kernel': ['poly', 'rbf', 'linear'],
#     'C': [0.1, 1, 10, 100],
#     'gamma': [1, 'scale', 'auto', 0.01, 0.1],
#     'epsilon': [0.01, 0.1, 0.5, 1],
#     'degree':[2]
# }

# grid_search_weighted_mape(SVR, param_grid_svr,  tickers)

# Best params: {'C': 10, 'epsilon': 0.5, 'gamma': 0.01, 'kernel': 'rbf'}
# Best weighted MAPE: 0.3043510536409729

[1/1] {'C': 10, 'epsilon': 0.5, 'gamma': 0.01, 'kernel': 'rbf'} - MAPE: 0.3044 (time: 1.5s)

Grid search completed in 0.03 minutes
Best params: {'C': 10, 'epsilon': 0.5, 'gamma': 0.01, 'kernel': 'rbf'}
Best weighted MAPE: 0.3043510536409729


({'C': 10, 'epsilon': 0.5, 'gamma': 0.01, 'kernel': 'rbf'}, 0.3043510536409729)

In [ ]:
from sklearn.linear_model import SGDRegressor
param_grid_sgd = {
    'loss': ['squared_error', 'huber', 'epsilon_insensitive'],
    'penalty': ['l2', 'l1', 'elasticnet'],
    'alpha': [1e-4, 1e-3, 1e-2],
    'learning_rate': ['constant', 'optimal', 'invscaling', 'adaptive'],
    'eta0': [1e-3, 1e-2, 1e-1],
    'epsilon': [0.1, 0.2, 0.5],
    'max_iter':[5000]
}
grid_search_weighted_mape(SGDRegressor, param_grid_sgd,  tickers)

# Grid search completed in 57.81 minutes
# Best params: {'alpha': 0.0001, 'epsilon': 0.2, 'eta0': 0.1, 'learning_rate': 'constant', 'loss': 'huber', 'max_iter': 5000, 'penalty': 'elasticnet'}
# Best weighted MAPE: 0.21570064125033528

In [ ]:
from sklearn.linear_model import Ridge, ElasticNet
param_grid_ridge = {
    'alpha': [0.01, 0.1, 1, 10, 100],
    'solver': ['auto', 'svd', 'cholesky', 'lsqr', 'saga'],
    'fit_intercept': [True, False]
}
grid_search_weighted_mape(Ridge, param_grid_ridge,  tickers)

# Grid search completed in 2.53 minutes
# Best params: {'alpha': 0.01, 'fit_intercept': False, 'solver': 'saga'}
# Best weighted MAPE: 0.377870828115418

param_grid_enet = {
    'alpha': [0.01, 0.1, 1, 10],
    'l1_ratio': [0.1, 0.5, 0.7, 0.9, 1],
    'fit_intercept': [True, False],
    'max_iter': [1000, 5000],
    'tol': [1e-4, 1e-3]
}

grid_search_weighted_mape(ElasticNet, param_grid_enet,  tickers)

# Grid search completed in 4.69 minutes
# Best params: {'alpha': 0.01, 'fit_intercept': False, 'l1_ratio': 1, 'max_iter': 5000, 'tol': 0.0001}
# Best weighted MAPE: 0.375367847063

KeyboardInterrupt: 